In [1]:
# pandas демо: от создания данных до продвинутых манипуляций
# цель: показать, как pandas упрощает работу с табличными данными

import numpy as np
import pandas as pd
import os

# для красивых графиков (опционально)
try:
    import matplotlib.pyplot as plt
    has_plt = True
except ImportError:
    has_plt = False

print("="*70)
print("pandas: работа с файлами, преобразования, группировки, слияния и аггрегации")
print("="*70)

# ------------------------------------------------------------
# 1. генерируем данные и сохраняем в файлы
# ------------------------------------------------------------

print("\n1. генерируем данные о продажах интернет-магазина")

np.random.seed(42)
n_rows = 100_000

# создаём dataframe
df = pd.DataFrame({
    'order_id': np.arange(1, n_rows + 1),
    'user_id': np.random.randint(1000, 2000, size=n_rows),
    'product_category': np.random.choice(['electronics', 'clothing', 'books', 'home', 'toys'], size=n_rows),
    'price': np.random.uniform(5, 500, size=n_rows).round(2),
    'quantity': np.random.randint(1, 6, size=n_rows),
    'order_date': pd.date_range('2023-01-01', periods=n_rows, freq='min'),  # уникальные даты с шагом в минуту
    'country': np.random.choice(['usa', 'canada', 'uk', 'germany', 'france'], size=n_rows, p=[0.5, 0.2, 0.1, 0.1, 0.1])
})

# добавляем колонку total = price * quantity
df['total'] = (df['price'] * df['quantity']).round(2)

print(f"создано {len(df):,} записей")
print("первые 5 строк:")
print(df.head())

# сохраняем в csv и excel
csv_file = 'sales_data.csv'
excel_file = 'sales_data.xlsx'

df.to_csv(csv_file, index=False, encoding='utf-8')
df.to_excel(excel_file, index=False, engine='openpyxl')

print(f"\nданные сохранены в {csv_file} и {excel_file}")

# ------------------------------------------------------------
# 2. читаем данные обратно и базовый анализ
# ------------------------------------------------------------

print("\n2. чтение данных из csv и первичный анализ")

df_read = pd.read_csv(csv_file, parse_dates=['order_date'])
print(f"загружено {len(df_read):,} строк, {df_read.shape[1]} колонок")
print("\nосновная статистика по числовым колонкам:")
print(df_read.describe())

print("\nтипы данных:")
print(df_read.dtypes)

# ------------------------------------------------------------
# 3. фильтрация и selection
# ------------------------------------------------------------

print("\n3. фильтрация данных")
# топ-10 самых дорогих заказов
top_orders = df_read.nlargest(10, 'total')
print("10 самых дорогих заказов:")
print(top_orders[['order_id', 'product_category', 'total', 'country']])

# заказы из категории electronics с суммой > 500
elec_high = df_read[(df_read['product_category'] == 'electronics') & (df_read['total'] > 500)]
print(f"\nзаказов electronics с суммой >500: {len(elec_high):,}")

# ------------------------------------------------------------
# 4. группировки и агрегации
# ------------------------------------------------------------

print("\n4. группировки и агрегации")

# выручка по категориям товаров
revenue_by_cat = df_read.groupby('product_category')['total'].agg(['sum', 'mean', 'count'])
revenue_by_cat.columns = ['total_revenue', 'avg_order_value', 'num_orders']
print("выручка по категориям:")
print(revenue_by_cat.sort_values('total_revenue', ascending=False))

# средний чек по стране и категории
pivot_country_cat = df_read.pivot_table(values='total', index='country', columns='product_category', aggfunc='mean', fill_value=0)
print("\nсредний чек по странам и категориям (pivot table):")
print(pivot_country_cat.round(2))

# ------------------------------------------------------------
# 5. обработка временных рядов
# ------------------------------------------------------------

print("\n5. анализ продаж по времени")

# добавляем колонки с годом, месяцем, днём недели
df_read['year'] = df_read['order_date'].dt.year
df_read['month'] = df_read['order_date'].dt.month
df_read['weekday'] = df_read['order_date'].dt.day_name()
df_read['hour'] = df_read['order_date'].dt.hour

# дневная выручка
daily_revenue = df_read.groupby(df_read['order_date'].dt.date)['total'].sum()
print("первые 10 дней выручки:")
print(daily_revenue.head(10))

# продажи по дням недели
sales_by_weekday = df_read.groupby('weekday')['total'].sum()
print("\nпродажи по дням недели:")
print(sales_by_weekday)

# ------------------------------------------------------------
# 6. работа с пропусками (намеренно создадим)
# ------------------------------------------------------------

print("\n6. работа с пропущенными данными")

# создаём копию и вносим пропуски
df_missing = df_read.copy()
mask = np.random.random(len(df_missing)) < 0.05  # 5% пропусков
df_missing.loc[mask, 'total'] = np.nan
df_missing.loc[mask, 'product_category'] = None

print(f"исходно строк: {len(df_missing)}")
print("количество пропусков:")
print(df_missing.isnull().sum())

# заполняем пропуски в total медианой по категории
df_missing['total'] = df_missing.groupby('product_category')['total'].transform(lambda x: x.fillna(x.median()))
# заполняем категорию модой
df_missing['product_category'].fillna(df_missing['product_category'].mode()[0], inplace=True)

print("\nпосле заполнения пропусков:")
print(df_missing.isnull().sum())

# ------------------------------------------------------------
# 7. слияние двух датафреймов (join/merge)
# ------------------------------------------------------------

print("\n7. пример слияния: добавляем информацию о пользователях")

# создаём справочник пользователей (1000 уникальных, но у нас user_id от 1000 до 2000)
users = pd.DataFrame({
    'user_id': range(1000, 2000),
    'age_group': np.random.choice(['18-25', '26-35', '36-50', '50+'], size=1000),
    'registration_date': pd.date_range('2020-01-01', periods=1000, freq='D')
})

# сливаем с основным df
df_merged = df_read.merge(users, on='user_id', how='left')
print(f"после merge: {df_merged.shape[1]} колонок")
print("первые 2 строки с пользовательскими данными:")
print(df_merged[['user_id', 'age_group', 'registration_date']].head(2))

# продажи по возрастным группам
sales_by_age = df_merged.groupby('age_group')['total'].sum().sort_values(ascending=False)
print("\nвыручка по возрастным группам:")
print(sales_by_age)

# ------------------------------------------------------------
# 8. применение функций к строкам и столбцам (apply, map)
# ------------------------------------------------------------

print("\n8. применение пользовательских функций")

# добавляем скидку: если total > 400, скидка 10%
df_read['discount'] = df_read['total'].apply(lambda x: x*0.1 if x > 400 else 0)
df_read['final_total'] = df_read['total'] - df_read['discount']

print("первые 5 строк с рассчитанной скидкой:")
print(df_read[['total', 'discount', 'final_total']].head())

# категоризация цен (map)
price_bins = {
    (0, 50): 'budget',
    (50, 150): 'mid',
    (150, 500): 'premium'
}

def price_category(price):
    for (low, high), cat in price_bins.items():
        if low < price <= high:
            return cat
    return 'luxury'

df_read['price_tier'] = df_read['price'].apply(price_category)
print("\nраспределение товаров по ценовым категориям:")
print(df_read['price_tier'].value_counts())

# ------------------------------------------------------------
# 9. производительность: query vs обычная фильтрация
# ------------------------------------------------------------

print("\n9. сравнение скорости фильтрации (обычный vs query)")

# обычная фильтрация
def filter_normal():
    return df_read[(df_read['product_category'] == 'electronics') & (df_read['total'] > 100)]

# query (использует numexpr, быстрее на больших данных)
def filter_query():
    return df_read.query("product_category == 'electronics' and total > 100")

import time
start = time.perf_counter()
res1 = filter_normal()
t_norm = time.perf_counter() - start

start = time.perf_counter()
res2 = filter_query()
t_query = time.perf_counter() - start

print(f"обычная фильтрация: {t_norm:.4f} сек")
print(f"query:              {t_query:.4f} сек")
print(f"ускорение:          {t_norm/t_query:.2f}x")
print("(разница может быть невелика на 100к строк, но на миллионах заметнее)")

# ------------------------------------------------------------
# 10. сохранение результатов в разных форматах
# ------------------------------------------------------------

print("\n10. сохраняем результаты агрегации и анализа")

# сохраняем сводку по категориям
revenue_by_cat.to_csv('revenue_by_category.csv')
# сохраняем дневную выручку в json
daily_revenue.reset_index().to_json('daily_revenue.json', orient='records', date_format='iso')
# сохраняем в parquet (быстро, сжато)
df_read.to_parquet('sales_data.parquet', index=False)

print("файлы сохранены: revenue_by_category.csv, daily_revenue.json, sales_data.parquet")

# если есть matplotlib, построим пару графиков
if has_plt:
    print("\n11. визуализация (matplotlib)")
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    
    # график 1: выручка по дням (первые 200 дней)
    daily_revenue.head(200).plot(ax=axes[0,0], title='daily revenue (first 200 days)')
    axes[0,0].set_ylabel('revenue')
    
    # график 2: продажи по категориям (pie)
    revenue_by_cat['total_revenue'].plot(kind='pie', ax=axes[0,1], autopct='%1.1f%%', title='revenue by category')
    
    # график 3: средний чек по странам (bar)
    pivot_country_cat.mean(axis=1).plot(kind='bar', ax=axes[1,0], title='average order value by country')
    
    # график 4: количество заказов по часам
    df_read.groupby('hour')['order_id'].count().plot(kind='line', marker='o', ax=axes[1,1], title='orders by hour')
    axes[1,1].set_xlabel('hour of day')
    axes[1,1].set_ylabel('number of orders')
    
    plt.tight_layout()
    plt.savefig('pandas_demo_plots.png', dpi=150)
    print("графики сохранены в pandas_demo_plots.png")
    plt.show()


print("\n" + "="*70)
print("итог: pandas позволяет")
print("- читать/писать файлы (csv, excel, parquet, json)")
print("- фильтровать, группировать, агрегировать, сводить таблицы")
print("- работать с временными рядами")
print("- обрабатывать пропуски")
print("- объединять датафреймы")
print("- применять пользовательские функции")
print("- строить графики (через matplotlib)")
print("- и всё это — быстро и с удобным синтаксисом")
print("="*70)

# чистим за собой (опционально)
# os.remove(csv_file)
# os.remove(excel_file)
# os.remove('revenue_by_category.csv')
# os.remove('daily_revenue.json')
# os.remove('sales_data.parquet')

pandas: работа с файлами, преобразования, группировки, слияния и аггрегации

1. генерируем данные о продажах интернет-магазина
создано 100,000 записей
первые 5 строк:
   order_id  user_id product_category   price  quantity          order_date  \
0         1     1102         clothing  394.96         2 2023-01-01 00:00:00   
1         2     1435             home  116.10         5 2023-01-01 00:01:00   
2         3     1860      electronics  287.49         5 2023-01-01 00:02:00   
3         4     1270            books  485.76         2 2023-01-01 00:03:00   
4         5     1106      electronics  265.68         3 2023-01-01 00:04:00   

   country    total  
0  germany   789.92  
1      usa   580.50  
2      usa  1437.45  
3       uk   971.52  
4      usa   797.04  

данные сохранены в sales_data.csv и sales_data.xlsx

2. чтение данных из csv и первичный анализ
загружено 100,000 строк, 8 колонок

основная статистика по числовым колонкам:
            order_id        user_id          price 

/var/folders/5d/rj2vhd7s3lgf7m08mvzp24xr0000gn/T/ipykernel_88053/2266901939.py:141: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_missing['product_category'].fillna(df_missing['product_category'].mode()[0], inplace=True)


ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.